<h1 style="margin-bottom: 5px;">Logo Classification</h1>
<h5 style="margin-top: 0; margin-bottom: 10px">
Artificial Neural Networks and Deep Learning D - Spring 2026<br>
Arman Kassam, Justin Kong, Shadab Sharif, and Tevin Park
</h5>
Logos are powerful visual signals that shape brand perception, influencing recognition, trust, and consumer behavior. In this project, we leverage deep learning to analyze and classify brand logos, uncovering patterns that link visual design to industry identity. By combining computer vision with structured labeling, we aim to explore how effectively a model can infer a company’s sector purely from its logo.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import os

from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/checkpoint.model.keras'
TUNED_CHECKPOINT_FILEPATH = '/kaggle/working/chkpts/tuned-checkpoint.model.keras'

In [ ]:
# dataset augmentation
strategy = tf.distribute.MirroredStrategy()

data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.1),
])

with strategy.scope():
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(23, activation='softmax', dtype='float32')(x)
    model = keras.Model(inputs, outputs)

In [ ]:
model.summary()

In [ ]:

labeled = pd.read_csv("/kaggle/input/datasets/jkong05/c-logo-labels/labels.csv")
labeled['sector'] = labeled['sector'].astype('category')
labeled['label_idx'] = labeled['sector'].cat.codes

classes = labeled['sector'].cat.categories
num_classes = len(classes)

def is_blurry(path, threshold=50):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True  # treat unreadable images as blurry
    return cv2.Laplacian(img, cv2.CV_64F).var() < threshold

# check a sample first to pick a good threshold
sample_paths = labeled['filepath'].sample(10, random_state=42).tolist()
for p in sample_paths:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    print(f"{score:.1f} — {p.split('/')[-1]}")

# filter the dataframe
tqdm.pandas()
labeled['blur_score'] = labeled['filepath'].progress_apply(
    lambda p: cv2.Laplacian(cv2.imread(p, cv2.IMREAD_GRAYSCALE), cv2.CV_64F).var() 
              if cv2.imread(p, cv2.IMREAD_GRAYSCALE) is not None else 0
)

before = len(labeled)
labeled = labeled[labeled['blur_score'] >= 100]
print(f"Removed {before - len(labeled)} blurry images, {len(labeled)} remaining")


train_df, temp_df = train_test_split(labeled, test_size=0.2, stratify=labeled["sector"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["sector"], random_state=42)


#added a function to make every image background white.
def standardize_background(path):
    
    img = Image.open(path)
    bg = Image.new('RGB', img.size, (255, 255, 255))
    
    if img.mode == 'RGBA':
        bg.paste(img, mask=img.split()[3])
    else:
        img = img.convert('RGB')
        corners = [img.getpixel((0, 0)),
                   img.getpixel((img.width-1, 0)),
                   img.getpixel((0, img.height-1)),
                   img.getpixel((img.width-1, img.height-1))]
        if np.mean([np.mean(c) for c in corners]) < 128:
            img = Image.fromarray(255 - np.array(img))
        bg.paste(img)
    
    bg = bg.resize((224, 224))
    return np.array(bg, dtype=np.float32)
    

'''def load_image(path, label):
    img = tf.py_function(
        lambda p: standardize_background(p.numpy().decode()),
        [path], tf.float32
    )
    img.set_shape([224, 224, 3])
    return img, label
'''
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    return tf.cast(img, tf.float32), label 
    
train_data = tf.data.Dataset.from_tensor_slices((train_df["filepath"].values, train_df["label_idx"].values))
train_data = train_data.shuffle(1000).map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

val_data = tf.data.Dataset.from_tensor_slices((val_df["filepath"].values, val_df["label_idx"].values))
val_data = val_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

test_data = tf.data.Dataset.from_tensor_slices((test_df["filepath"].values, test_df["label_idx"].values))
test_data = test_data.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)

In [ ]:
sample_paths = labeled['filepath'].sample(10, random_state=42).tolist()
for p in sample_paths:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    print(f"{score:.1f} — {p.split('/')[-1]}")

In [ ]:
top10_blurry = labeled.nlargest(10, 'blur_score')

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, (_, row) in enumerate(top10_blurry.iterrows()):
    img = Image.open(row['filepath']).convert('RGB').resize((224, 224))
    axes[i//5, i%5].imshow(img)
    axes[i//5, i%5].set_title(f"{row['sector']}\nscore: {row['blur_score']:.1f}")
    axes[i//5, i%5].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(labeled['blur_score'], bins=100, log=True)
plt.xlabel('Blur Score')
plt.ylabel('Count (log scale)')
plt.title('Blur Score Distribution')
plt.axvline(x=30, color='red', label='threshold=30')
plt.legend()
plt.show()

print(labeled['blur_score'].describe())
print(f"\nBelow 10: {(labeled['blur_score'] < 10).sum()}")
print(f"Below 20: {(labeled['blur_score'] < 20).sum()}")
print(f"Below 50: {(labeled['blur_score'] < 50).sum()}")
print(f"Below 100: {(labeled['blur_score'] < 100).sum()}")

In [ ]:
# look at images scoring between 30-50
borderline = labeled[labeled['blur_score'] < 50]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i, (_, row) in enumerate(borderline.sample(12, random_state=42).iterrows()):
    img = Image.open(row['filepath']).convert('RGB').resize((224, 224))
    axes[i//4, i%4].imshow(img)
    axes[i//4, i%4].set_title(f"score: {row['blur_score']:.1f}")
    axes[i//4, i%4].axis('off')

plt.tight_layout()
plt.show()

In [ ]:

# REPLACE the model building cell with this:
#strategy = tf.distribute.MirroredStrategy()
#with strategy.scope():
#    base_model = keras.models.load_model("/kaggle/input/models/shadabsharif/14class/keras/default/1/phase2_14cls.keras")

#inputs = base_model.input
#x = base_model.get_layer('global_average_pooling2d').output
#x = base_model.get_layer('dropout')(x)
#outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
#model = keras.Model(inputs, outputs)

In [ ]:
from PIL import Image

sample_paths = labeled['filepath'].sample(5, random_state=42).tolist()

fig, axes = plt.subplots(5, 2, figsize=(8, 20))
for i, path in enumerate(sample_paths):
    # original
    orig = Image.open(path).convert('RGB').resize((224, 224))
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title('Original')
    axes[i, 0].axis('off')
    
    # processed
    processed = standardize_background(path)
    axes[i, 1].imshow(processed.astype(np.uint8))
    axes[i, 1].set_title('Standardized')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labeled['label_idx']),
    y=labeled['label_idx']
)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

In [ ]:
with strategy.scope():
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.SparseTopKCategoricalAccuracy(k=4, name='top_4_SparseTopKCategoricalAccuracy')]
    )

callbacks = [
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_FILEPATH,
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

In [ ]:

#loading base model from checkpoint for fine-tuning
    #- uncomment the line below and re-run following cells for base model training

#strategy = tf.distribute.MirroredStrategy()
#with strategy.scope():
#    model = keras.models.load_model("/kaggle/input/models/shadabsharif/14class/keras/default/1/phase2_14cls.keras")


In [ ]:
 H = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

In [ ]:
model.save('/kaggle/working/phase1_23cls.keras')

In [ ]:
def create_accuracy_loss_plot(H):
    acc = H.history['accuracy']
    val_acc = H.history['val_accuracy']
    loss = H.history['loss']
    val_loss = H.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, acc, label='Train')
    ax1.plot(epochs, val_acc, label='Val')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(epochs, loss, label='Train')
    ax2.plot(epochs, val_loss, label='Val')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    fig.savefig('/kaggle/working/accuracy_loss.png', bbox_inches='tight', dpi=150)
    plt.show()

def create_confusion_matrix(model, test_data, classes):
    preds = model.predict(test_data).argmax(axis=1)
    true_labels = np.concatenate([y for _, y in test_data], axis=0)

    cm = confusion_matrix(true_labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)

    fig, ax = plt.subplots(figsize=(16, 16))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
    plt.tight_layout()
    fig.savefig('/kaggle/working/confusion_matrix.png', bbox_inches='tight', dpi=150)
    plt.show()
    

def create_heatmap(img_array, model):
    base = model.get_layer('efficientnetb0')

    inner_grad_model = tf.keras.Model(
        inputs=base.inputs,
        outputs=[base.get_layer('top_activation').output, base.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, base_out = inner_grad_model(img_array, training=False)
        tape.watch(conv_outputs)

        x = model.get_layer('global_average_pooling2d')(base_out)
        x = model.get_layer('dropout')(x, training=False)
        predictions = model.get_layer('dense')(x)

        pred_idx = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_idx]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_idx.numpy()


def display_heatmap(img_path, true_idx, model, classes):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img_array = tf.expand_dims(tf.cast(img, tf.float32), 0)

    heatmap, pred_idx = create_heatmap(img_array, model)

    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    orig = np.uint8(img.numpy())
    overlay = cv2.addWeighted(orig, 0.6, heatmap, 0.4, 0)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
    ax1.imshow(orig)
    ax1.set_title('Original') 
    ax1.axis('off')
    ax2.imshow(heatmap) 
    ax2.set_title('Heatmap') 
    ax2.axis('off')
    ax3.imshow(overlay)
    ax3.set_title(f'Pred: {classes[pred_idx]}\nTrue: {classes[true_idx]}')
    ax3.axis('off')
    plt.tight_layout()
    fig.savefig(f'/kaggle/working/heatmap_{os.path.basename(img_path)}.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
create_accuracy_loss_plot(H)

In [ ]:
create_confusion_matrix(model, test_data, classes)

In [ ]:
for img_path, true_idx in zip(test_df['filepath'].iloc[:5], test_df['label_idx'].iloc[:5]):
    display_heatmap(img_path, true_idx, model, classes)

In [ ]:
# tuning

base = model.get_layer('efficientnetb0')

unfreeze_blocks = ('block3', 'block4', 'block5', 'block6', 'block7', 'top')

for layer in base.layers:
    if layer.name.startswith(unfreeze_blocks) and not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True
    else:
        layer.trainable = False

with strategy.scope():
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-6),
        loss='sparse_categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top_3_SparseTopKCategoricalAccuracy'),
        ]
    )

tuned_callbacks = [
    keras.callbacks.ModelCheckpoint(
        TUNED_CHECKPOINT_FILEPATH,
        monitor="val_loss",
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]


In [ ]:
#H2 = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=tuned_callbacks, class_weight=class_weight_dict)

In [ ]:
#create_accuracy_loss_plot(H2)

In [ ]:
#create_confusion_matrix(model, test_data, classes)

In [ ]:
#for img_path, true_idx in zip(test_df['filepath'].iloc[:5], test_df['label_idx'].iloc[:5]):
   #display_heatmap(img_path, true_idx, model, classes)

In [ ]:
#model.save('/kaggle/working/phase2_14cls_frozen_color.keras')